<a href="https://colab.research.google.com/github/fairclj/HMI-2026-Demo/blob/main/High_Dimensional_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **High-Dimensional Visualization**

In [7]:
# @title
# ============================================================
# STEP 1 — IMPORT LIBRARIES
# ============================================================
import io
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from google.colab import files

# Use Colab renderer
pio.renderers.default = "colab"

# ============================================================
# STEP 2 — ASK USER TO UPLOAD CSV
# ============================================================
print("Please upload a CSV dataset file.")

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file was uploaded.")

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"\nSuccessfully loaded: {filename}")
print(f"Dataset shape: {df.shape}")

# ============================================================
# STEP 3 — SHOW AVAILABLE COLUMNS
# ============================================================
print("\nAvailable columns in your dataset:")
for col in df.columns:
    print(f" - {col}")

# ============================================================
# STEP 4 — ASK USER FOR ID AND OUTCOME COLUMN NAMES
# ============================================================
id_col = input("\nType the exact name of the ID column: ").strip()
outcome_col = input("Type the exact name of the outcome column: ").strip()

if id_col not in df.columns:
    raise ValueError(f"ID column '{id_col}' was not found in the dataset.")

if outcome_col not in df.columns:
    raise ValueError(f"Outcome column '{outcome_col}' was not found in the dataset.")

if id_col == outcome_col:
    raise ValueError("The ID column and outcome column must be different.")

# ============================================================
# STEP 5 — PREPROCESS FEATURES
# ============================================================
from IPython.display import display, HTML

display(HTML("""
<style>
.container {
    width: 100% !important;
}
</style>
"""))

student_ids = df[id_col].astype(str)
outcomes = df[outcome_col].astype(str)

X = df.drop(columns=[id_col, outcome_col]).copy()

if X.shape[1] == 0:
    raise ValueError("No feature columns remain after removing the ID and outcome columns.")

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

if len(categorical_cols) > 0:
    encoder = OrdinalEncoder()
    X[categorical_cols] = encoder.fit_transform(X[categorical_cols])

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

all_missing_cols = X.columns[X.isna().all()].tolist()
if len(all_missing_cols) > 0:
    print("\nDropping columns with no usable information:")
    for col in all_missing_cols:
        print(f" - {col}")
    X = X.drop(columns=all_missing_cols)

if X.shape[1] == 0:
    raise ValueError("No usable feature columns remain after preprocessing.")

X = X.fillna(X.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nFeature matrix shape after preprocessing: {X_scaled.shape}")
print("Features used:")
for col in X.columns:
    print(f" - {col}")

# ============================================================
# STEP 6 — CREATE HIGH-DIMENSIONAL-STYLE SIMILARITY SPACE
# ============================================================
n_samples = X_scaled.shape[0]

perplexity_value = min(30, max(5, n_samples // 3))
if perplexity_value >= n_samples:
    perplexity_value = max(1, n_samples - 1)

K = rbf_kernel(X_scaled, gamma=0.5)

D = 1 - K
D = (D + D.T) / 2
np.fill_diagonal(D, 0)

coords = TSNE(
    n_components=3,
    metric="precomputed",
    init="random",
    perplexity=perplexity_value,
    random_state=42
).fit_transform(D)

# ============================================================
# STEP 7 — BUILD COLOR MAP AUTOMATICALLY
# ============================================================
unique_outcomes = list(pd.Series(outcomes).astype(str).unique())

palette = [
    "#22c55e", "#ef4444", "#60a5fa", "#fbbf24",
    "#a78bfa", "#f472b6", "#22d3ee", "#fb7185",
    "#14b8a6", "#eab308"
]

color_map = {
    outcome: palette[i % len(palette)]
    for i, outcome in enumerate(unique_outcomes)
}

colors = outcomes.map(color_map)

# ============================================================
# STEP 8 — BUILD HOVER TEXT WITH FULL PROFILE
# ============================================================
hover_text = []

for i in range(len(df)):
    lines = [
        f"<b>{id_col}:</b> {df.iloc[i][id_col]}",
        f"<b>{outcome_col}:</b> {df.iloc[i][outcome_col]}",
        "<br><b>Profile</b>"
    ]
    for col in df.columns:
        if col not in [id_col, outcome_col]:
            lines.append(f"<b>{col}:</b> {df.iloc[i][col]}")
    hover_text.append("<br>".join(lines))

# ============================================================
# STEP 9 — CREATE INTERACTIVE 3D VISUAL
# ============================================================
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=coords[:, 0],
    y=coords[:, 1],
    z=coords[:, 2],
    mode="markers",
    marker=dict(
        size=4,
        color=colors,
        opacity=0.85
    ),
    text=hover_text,
    hovertemplate="%{text}<extra></extra>",
    showlegend=False
))

for outcome in unique_outcomes:
    fig.add_trace(go.Scatter3d(
        x=[None],
        y=[None],
        z=[None],
        mode="markers",
        marker=dict(size=6, color=color_map[outcome]),
        name=str(outcome),
        showlegend=True
    ))

fig.update_layout(
    title="High-Dimensional Similarity Map",
    paper_bgcolor="black",
    plot_bgcolor="black",
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="black",
        camera=dict(
            eye=dict(x=1.8, y=1.8, z=0.9)
        )
    ),
    font=dict(color="white"),
    width=1000,
    height=700,
    legend=dict(
        bgcolor="rgba(0,0,0,0.35)",
        font=dict(color="white")
    )
)

fig.show()

# ============================================================
# STEP 10 — RELIABLE STUDENT LOOKUP
# ============================================================
print(f"\nTo inspect a specific record, type a value from the '{id_col}' column.")
print("Type 'exit' to stop.\n")

while True:
    selected_id = input(f"Enter {id_col}: ").strip()

    if selected_id.lower() == "exit":
        break

    match = df[df[id_col].astype(str) == selected_id]

    if len(match) == 0:
        print("No matching record found.\n")
    else:
        print("\nFULL PROFILE")
        print("-" * 60)
        print(match.to_string(index=False))
        print()

Please upload a CSV dataset file.


Saving synthetic_step1_biostat_dataset.csv to synthetic_step1_biostat_dataset (2).csv

Successfully loaded: synthetic_step1_biostat_dataset (2).csv
Dataset shape: (300, 23)

Available columns in your dataset:
 - student_id
 - cohort_year
 - exam_month
 - institution_region
 - program_type
 - age_years
 - prior_statistics_course
 - preclinical_gpa
 - undergrad_quant_score_pct
 - study_hours_biostats_per_week
 - study_days_before_exam
 - practice_exam_count
 - qbank_biostats_questions_completed
 - qbank_biostats_accuracy_pct
 - nbme_baseline_biostats_pct
 - epidemiology_quiz_pct
 - average_question_time_sec
 - sleep_hours_night_before_exam
 - test_anxiety_score_1_10
 - self_reported_confidence_1_10
 - biostat_performance_index_pct
 - step1_outcome
 - risk_band

Type the exact name of the ID column: student_id
Type the exact name of the outcome column: step1_outcome



Feature matrix shape after preprocessing: (300, 21)
Features used:
 - cohort_year
 - exam_month
 - institution_region
 - program_type
 - age_years
 - prior_statistics_course
 - preclinical_gpa
 - undergrad_quant_score_pct
 - study_hours_biostats_per_week
 - study_days_before_exam
 - practice_exam_count
 - qbank_biostats_questions_completed
 - qbank_biostats_accuracy_pct
 - nbme_baseline_biostats_pct
 - epidemiology_quiz_pct
 - average_question_time_sec
 - sleep_hours_night_before_exam
 - test_anxiety_score_1_10
 - self_reported_confidence_1_10
 - biostat_performance_index_pct
 - risk_band



To inspect a specific record, type a value from the 'student_id' column.
Type 'exit' to stop.



KeyboardInterrupt: Interrupted by user